## **SECTION A: SETUP**

**Objective:** Initialize the environment by loading the necessary Python libraries and authenticating the Google Maps API key.

**Tools Used:**
- `pandas` for data manipulation.
- `googlemaps` for API requests.
- `dotenv` to securely load the API key from the `.env` file.

In [1]:
import os
import pandas as pd
import googlemaps
from dotenv import load_dotenv

# Load API Key
load_dotenv()
API_KEY = os.getenv("GOOGLE_MAPS_API_KEY")

if not API_KEY:
    raise ValueError("❌ API Key not found. Check your .env file.")

# Start Google Maps Client
gmaps = googlemaps.Client(key=API_KEY)

print("✅ Setup complete. Ready to fetch data.")

✅ Setup complete. Ready to fetch data.


## **SECTION B: FINDING THE RESORTS**

**Objective:** Search Google Places for waterfront resorts across the Kenyan coast, lakes, and safari rivers. 

**Data Collected:** 
- `ResortName`
- `Address` & `Location` (for the map)
- `Latitude` & `Longitude`
- `GoogleRating` (the overall score)
- `PlaceID` (to fetch reviews later)

In [2]:
# The search queries covering coast, lakes, and rivers
search_queries = [
    "resorts in Diani Beach Kenya",
    "beach resorts in Mombasa Shanzu Nyali",
    "resorts in Watamu and Malindi Kenya",
    "hotels in Lamu Archipelago Shela",
    "resorts around Lake Naivasha Kenya",
    "hotels near Lake Nakuru Kenya",
    "lodges around Lake Elementaita",
    "waterfront hotels in Kisumu Lake Victoria",
    "lodges along Mara River Masai Mara",
    "river lodges Samburu Ewaso Nyiro"
]

all_resorts = {}

print("🔍 Searching for resorts...")

for query in search_queries:
    try:
        response = gmaps.places(query=query)
        for place in response.get("results", []):
            place_id = place.get("place_id")
            
            # Only add if we haven't seen it before
            if place_id and place_id not in all_resorts:
                
                # Clean up the location name
                region = "Other"
                q_lower = query.lower()
                if "diani" in q_lower: region = "Coast - Diani & South"
                elif "mombasa" in q_lower or "nyali" in q_lower: region = "Coast - Mombasa & North"
                elif "watamu" in q_lower or "malindi" in q_lower: region = "Coast - Watamu & Malindi"
                elif "lamu" in q_lower: region = "Coast - Lamu"
                elif "naivasha" in q_lower: region = "Inland Lake - Naivasha"
                elif "nakuru" in q_lower: region = "Inland Lake - Nakuru"
                elif "elementaita" in q_lower: region = "Inland Lake - Elementaita"
                elif "kisumu" in q_lower or "victoria" in q_lower: region = "Inland Lake - Victoria"
                elif "mara" in q_lower or "samburu" in q_lower or "river" in q_lower: region = "Safari River"

                all_resorts[place_id] = {
                    "ResortName": place.get("name"),
                    "Address": place.get("formatted_address"),
                    "Location": region,
                    "Latitude": place.get("geometry", {}).get("location", {}).get("lat"),
                    "Longitude": place.get("geometry", {}).get("location", {}).get("lng"),
                    "GoogleRating": place.get("rating"),
                    "PlaceID": place_id
                }
    except Exception as e:
        print(f"⚠️ Error with query '{query}': {e}")

# Convert to a DataFrame
df = pd.DataFrame(list(all_resorts.values()))

# Remove the duplicate Mara River Lodge in Nairobi (if it appears)
bad_address = "Emabakasi,Airport North Road Behind Rubis Petrol Station, MWH3+899, Nairobi, Kenya"
df = df[df["Address"] != bad_address]

print(f"✅ Found {len(df)} unique resorts.")
df.head()

🔍 Searching for resorts...
✅ Found 169 unique resorts.


,ResortName,Address,Location,Latitude,Longitude,GoogleRating,PlaceID
0,Diani Reef Beach Resort & Spa,"Diani Beach Road, Diani Beach, Kenya",Coast - Diani & South,-4.278903,39.593257,4.3,ChIJJ_7_UB5GQBgRBjFxDV0y62o
1,Swahili Beach Resort,"Ukunda, Diani Beach Road, Kwale, Kenya",Coast - Diani & South,-4.292183,39.586675,4.4,ChIJlTS5haBIQBgRTH2fzpOoC5E
2,Diamonds Leisure Beach & Golf Resort,"Diani Road, South, Coast, Kwale, Kenya",Coast - Diani & South,-4.281405,39.592070,4.4,ChIJsaY6hz1GQBgRdjvawvwqcpk
3,Coral Beach Resort,"Diani Beach, Kenya",Coast - Diani & South,-4.289194,39.589740,4.1,ChIJkag91qBHQBgRQTkD5BdAgnc
4,Southern Palms Beach Resort,"Plot LR 13430, Diani Beach Road Diani, Ukunda,...",Coast - Diani & South,-4.268264,39.597005,4.3,ChIJNeN4-kRGQBgRVxlknZWsFL8


## **SECTION C: FIRST CLEANING STEP**

**Objective:** Remove duplicates, fix invalid locations, and drop any resorts that are missing critical data (like a `GoogleRating`).

In [3]:
# 1. Remove the fake "Mara River Lodge" in Nairobi (if it snuck back in)
# We use a partial string match to catch it just in case Google changed the address format
df = df[~df["Address"].str.contains("Nairobi", case=False, na=False) | ~df["ResortName"].str.contains("Mara River Lodge", case=False, na=False)]

# 2. Drop any resort that doesn't have a Google Rating
# (You can't rank a hotel if it has no score!)
df = df.dropna(subset=["GoogleRating"])

# 3. Check the final count
print(f"✅ Cleaned dataset now has {len(df)} resorts.")
print(df["Location"].value_counts())

✅ Cleaned dataset now has 168 resorts.
Location
Safari River                 22
Coast - Diani & South        20
Coast - Watamu & Malindi     20
Coast - Lamu                 20
Inland Lake - Naivasha       20
Inland Lake - Nakuru         20
Inland Lake - Victoria       20
Inland Lake - Elementaita    19
Coast - Mombasa & North       7
Name: count, dtype: int64


## **SECTION D: FETCHING REVIEWS**

**Objective:** Use the `PlaceID` to fetch the `ReviewCount` and `ReviewText` for each resort. This text will be used to power the future AI review summary, helping travelers understand what a hotel is truly known for.

In [4]:
def get_reviews(place_id):
    try:
        # Fetch total reviews and up to 5 review texts
        details = gmaps.place(
            place_id=place_id, 
            fields=["user_ratings_total", "reviews"]
        )["result"]
        
        count = details.get("user_ratings_total", 0)
        reviews_list = details.get("reviews", [])
        
        # Join the reviews into one string, separated by a delimiter
        text = " || ".join([r.get("text", "") for r in reviews_list])
        
        return pd.Series({"ReviewCount": count, "ReviewText": text})
    except Exception:
        return pd.Series({"ReviewCount": 0, "ReviewText": ""})

print("⏳ Fetching reviews for 168 resorts. This will take a few minutes...")

# Apply to every row and merge
reviews_df = df["PlaceID"].apply(get_reviews)
df = pd.concat([df, reviews_df], axis=1)

# Clean up any line breaks in the text so Tableau doesn't break
df["ReviewText"] = df["ReviewText"].fillna("").str.replace(r"[\r\n]+", " ", regex=True)

print("✅ Reviews fetched and cleaned.")
print(df[["ResortName", "ReviewCount"]].head())

⏳ Fetching reviews for 168 resorts. This will take a few minutes...
✅ Reviews fetched and cleaned.
                             ResortName  ReviewCount
0         Diani Reef Beach Resort & Spa         2759
1                  Swahili Beach Resort         3692
2  Diamonds Leisure Beach & Golf Resort         4344
3                    Coral Beach Resort          180
4           Southern Palms Beach Resort         3310


## **SECTION E: SAVING THE CLEAN DATASET**

**Objective:** Save the final, cleaned dataset to a new CSV file. This file will be the single source of truth for the Tableau dashboard.

**Final Columns Kept:**
- `ResortName`, `Address`, `Location`, `Latitude`, `Longitude`
- `GoogleRating`, `ReviewCount`, `ReviewText`
- `PlaceID`

In [5]:
# Save to a brand new CSV file
output_path = "Data/kenya_resorts_clean.csv"
df.to_csv(output_path, index=False)

print(f"✅ Dataset saved to {output_path}")
print(f"Final Shape: {df.shape[0]} rows and {df.shape[1]} columns.")
print(f"Columns: {list(df.columns)}")

✅ Dataset saved to Data/kenya_resorts_clean.csv
Final Shape: 168 rows and 9 columns.
Columns: ['ResortName', 'Address', 'Location', 'Latitude', 'Longitude', 'GoogleRating', 'PlaceID', 'ReviewCount', 'ReviewText']


## **SECTION G: GEOGRAPHIC CLEANING**

**Objective:** The original `Location` column mixes abstract categories with real places (e.g., "Coast - Diani & South"). To make the dashboard intuitive for travelers, we are splitting this into two clean columns:
- `County`: The official Kenyan county (e.g., Kwale, Nakuru, Lamu).
- `Place`: The specific destination (e.g., Diani Beach, Naivasha, Watamu).

In [9]:
def get_clean_location(row):
    loc = str(row["Location"]).lower()
    address = str(row["Address"]).lower()
    
    # Defaults
    county = "Unknown"
    place = "Unknown"
    
    if "diani" in loc or "diani" in address:
        county = "Kwale"
        place = "Diani Beach"
    elif "mombasa" in loc:
        county = "Mombasa"
        place = "Mombasa"
    elif "watamu" in loc or "watamu" in address:
        county = "Kilifi"
        place = "Watamu"
    elif "malindi" in loc or "malindi" in address:
        county = "Kilifi"
        place = "Malindi"
    elif "lamu" in loc:
        county = "Lamu"
        place = "Lamu"
    elif "naivasha" in loc:
        county = "Nakuru"
        place = "Naivasha"
    elif "elementaita" in loc:
        county = "Nakuru"
        place = "Elementaita"
    elif "nakuru" in loc:
        county = "Nakuru"
        place = "Nakuru"
    elif "victoria" in loc or "kisumu" in address:
        county = "Kisumu"
        place = "Kisumu"
    elif "safari river" in loc:
        if "mara" in address:
            county = "Narok"
            place = "Masai Mara"
        elif "samburu" in address:
            county = "Samburu"
            place = "Samburu"
    
    return pd.Series({"County": county, "Place": place})

# Apply the function to create the new columns
clean_locs = df.apply(get_clean_location, axis=1)
df = pd.concat([df, clean_locs], axis=1)

# Drop the old messy Location column
df = df.drop(columns=["Location"])

# Save the updated clean dataset
df.to_csv("Data/kenya_resorts_clean.csv", index=False)

print("✅ Geographic columns added.")
print(df[["ResortName", "County", "Place"]].head())

✅ Geographic columns added.
                             ResortName County        Place
0         Diani Reef Beach Resort & Spa  Kwale  Diani Beach
1                  Swahili Beach Resort  Kwale  Diani Beach
2  Diamonds Leisure Beach & Golf Resort  Kwale  Diani Beach
3                    Coral Beach Resort  Kwale  Diani Beach
4           Southern Palms Beach Resort  Kwale  Diani Beach


## **SECTION H: SPATIAL JOIN WITH COUNTY SHAPEFILE**

**Objective:** Replace manual text-based location guessing with a geometric spatial join. Every resort's latitude/longitude is tested against the Kenya county polygons to assign an official `County`.

**Why:** The previous method failed for 7 resorts because their address text didn't match any keyword. A spatial join is reliable because it uses coordinates, not words.

In [12]:
import geopandas as gpd
import pandas as pd

# 1. Load the county shapefile from the new folder
counties = gpd.read_file("Data/kenyan-counties/County.shp")

# 2. Check the columns
print("County file columns:")
print(list(counties.columns))
print("\nFirst 3 rows:")
print(counties.head(3))

County file columns:
['OBJECTID', 'AREA', 'PERIMETER', 'COUNTY3_', 'COUNTY3_ID', 'COUNTY', 'Shape_Leng', 'Shape_Area', 'geometry']

First 3 rows:
   OBJECTID   AREA  PERIMETER  COUNTY3_  COUNTY3_ID    COUNTY  Shape_Leng  \
0         1  5.677     15.047       2.0         1.0   Turkana   15.046838   
1         2  6.177     11.974       3.0         2.0  Marsabit   11.974165   
2         3  2.117      7.355       4.0         3.0   Mandera    7.355154   

   Shape_Area                                           geometry  
0    5.676985  POLYGON ((35.79593 5.34449, 35.79659 5.34468, ...  
1    6.176831  POLYGON ((36.05061 4.45622, 36.23184 4.45124, ...  
2    2.117196  POLYGON ((41.62133 3.97673, 41.62272 3.9786, 4...  


## **SECTION I: SPATIAL JOIN (FIXING UNKNOWNS)**

**Objective:** Replace the manual text-based `County` column with a reliable spatial join. We will convert each resort's `Latitude` and `Longitude` into a geometric point and check which county polygon it falls inside.

In [13]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# 1. Load the clean dataset
df = pd.read_csv("Data/kenya_resorts_clean.csv")

# 2. Load the counties shapefile and ensure it uses standard WGS84 coordinates
counties = gpd.read_file("Data/kenyan-counties/County.shp")
counties = counties.to_crs(epsg=4326)

# 3. Create Point geometry for each resort
geometry = [Point(xy) for xy in zip(df["Longitude"], df["Latitude"])]
resorts_gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

# 4. Perform the spatial join
joined = gpd.sjoin(resorts_gdf, counties[['COUNTY', 'geometry']], how="left", predicate="intersects")

# 5. Update the main dataframe
# The shapefile uses 'COUNTY' in uppercase, so we drop the old manual 'County' and use the official one
df['County'] = joined['COUNTY']

# Check if any resorts failed the join (e.g., they are slightly offshore)
null_count = df['County'].isna().sum()
print(f"Resorts without a spatial county: {null_count}")

# If there are any nulls, fill them with "Unknown"
df['County'] = df['County'].fillna("Unknown")

# 6. Save the updated dataset
df.to_csv("Data/kenya_resorts_clean.csv", index=False)

print("✅ Spatial join complete. Final county counts:")
print(df["County"].value_counts())

Resorts without a spatial county: 31
✅ Spatial join complete. Final county counts:
County
Nakuru     59
Unknown    31
Kisumu     20
Narok      19
Lamu       13
Kwale      11
Kilifi      8
Mombasa     4
Kajiado     2
Isiolo      1
Name: count, dtype: int64


In [14]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# 1. Reload the clean dataset
df = pd.read_csv("Data/kenya_resorts_clean.csv")

# 2. Reload the shapefile
counties = gpd.read_file("Data/kenyan-counties/County.shp")
counties = counties.to_crs(epsg=4326)

# 3. Recreate the points
geometry = [Point(xy) for xy in zip(df["Longitude"], df["Latitude"])]
resorts_gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

# 4. Perform the standard join (intersects)
joined = gpd.sjoin(resorts_gdf, counties[['COUNTY', 'geometry']], how="left", predicate="intersects")

# 5. Find the ones that failed (the 31 unknowns)
unmatched_mask = joined['COUNTY'].isna()
unmatched_resorts = resorts_gdf[unmatched_mask]

# 6. Perform the nearest join ONLY for the unmatched ones
if not unmatched_resorts.empty:
    nearest_join = gpd.sjoin_nearest(unmatched_resorts, counties[['COUNTY', 'geometry']], how="left")
    
    # 7. Update the joined dataframe with the nearest county results
    joined.loc[unmatched_mask, 'COUNTY'] = nearest_join['COUNTY'].values

# 8. Update the main dataframe
df['County'] = joined['COUNTY']

# 9. Final check
print(f"Resorts without a county after Nearest Join: {df['County'].isna().sum()}")
print("\nFinal County counts:")
print(df["County"].value_counts())

# 10. Save the updated dataset
df.to_csv("Data/kenya_resorts_clean.csv", index=False)
print("✅ Saved to Data/kenya_resorts_clean.csv")

Resorts without a county after Nearest Join: 0

Final County counts:
County
Nakuru     59
Kwale      20
Kilifi     20
Lamu       20
Kisumu     20
Narok      19
Mombasa     7
Kajiado     2
Isiolo      1
Name: count, dtype: int64
✅ Saved to Data/kenya_resorts_clean.csv


c:\Users\User\desktop\GenData-Capstone-Project\venv\Lib\site-packages\geopandas\array.py:438: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


## **SECTION J: EXTRACTING STANDOUT THEMES**

**Objective:** Scan the `ReviewText` to find the most frequently mentioned theme (e.g., Views, Service, Food) for each resort. This creates a `StandoutTheme` column to show the user *why* a resort is highly rated.

In [3]:
import pandas as pd
import re
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# 1. Load your dataset
df = pd.read_csv("Data/kenya_resorts_clean.csv")

# 2. Initialize VADER
analyzer = SentimentIntensityAnalyzer()

# 3. Define themes
themes = {
    "Views & Scenery": ["view", "scenery", "beach", "ocean", "lake", "sunset"],
    "Service": ["staff", "service", "friendly", "hospitality", "welcoming"],
    "Food": ["food", "breakfast", "dinner", "meal", "buffet", "restaurant"],
    "Rooms": ["room", "bed", "bathroom", "shower", "suite"],
    "Amenities": ["pool", "spa", "wifi", "internet", "gym", "bar"]
}

def get_aspect_sentiment_theme(text):
    if pd.isna(text) or text == "":
        return "General"
    
    # Split by punctuation AND conjunctions (but, however, although) to isolate clauses
    clauses = re.split(r'[.!?]|,\s*but\s*|\s+but\s+|;\s*|,\s*however\s*', str(text), flags=re.IGNORECASE)
    
    theme_scores = {theme: 0 for theme in themes}
    
    for clause in clauses:
        if not clause.strip():
            continue
            
        clause_lower = clause.lower()
        
        for theme, keywords in themes.items():
            # If the clause mentions the theme
            if any(re.search(r'\b' + kw + r'\b', clause_lower) for kw in keywords):
                sentiment = analyzer.polarity_scores(clause)['compound']
                
                # ONLY count it if the clause is positive
                if sentiment > 0.05:
                    theme_scores[theme] += 1
                    
    best_theme = max(theme_scores, key=theme_scores.get)
    
    if theme_scores[best_theme] == 0:
        return "General"
        
    return best_theme

print("Running smarter sentiment analysis...")
df["StandoutTheme"] = df["ReviewText"].apply(get_aspect_sentiment_theme)

# Save the updated dataset
df.to_csv("Data/kenya_resorts_clean.csv", index=False)

print("Done! Dataset updated.")
print(df["StandoutTheme"].value_counts())

Running smarter sentiment analysis...
Done! Dataset updated.
StandoutTheme
Service            105
Views & Scenery     34
Food                21
Rooms                3
General              3
Amenities            2
Name: count, dtype: int64


In [4]:
# Let's verify the logic.
# We will look at a resort that has mixed reviews, and a resort that has terrible reviews.

# 1. Diani Reef Beach Resort & Spa has a mix of glowing reviews and one terrible review.
# 2. Diamond Luxury Homes watamu has mostly terrible reviews.

verify_resorts = ["Diani Reef Beach Resort & Spa", "Diamond Luxury Homes watamu", "Swahili Beach Resort"]

for resort in verify_resorts:
    row = df[df["ResortName"] == resort]
    if not row.empty:
        print(f"\n{'='*50}")
        print(f"RESORT: {resort}")
        print(f"ASSIGNED THEME: {row['StandoutTheme'].values[0]}")
        print(f"{'-'*50}")
        # Print the first 300 characters of the review text so it fits on screen
        print("SAMPLE REVIEW TEXT:")
        print(str(row['ReviewText'].values[0])[:300] + "...\n")


RESORT: Diani Reef Beach Resort & Spa
ASSIGNED THEME: Service
--------------------------------------------------
SAMPLE REVIEW TEXT:
I absolutely loved the place. Right from the reservation which was done well by Mrs Sandra to the service everything was done well. We initially booked a garden view but upon request we were switched to sea view which was so thoughtful. The rooms are clean and the buffet had a variety of options wit...


RESORT: Diamond Luxury Homes watamu
ASSIGNED THEME: Service
--------------------------------------------------
SAMPLE REVIEW TEXT:
Do not rent this place!! The property is in very poor condition. We had repeated power and water outages, non-working air conditioning, and a serious problem with rodents and cockroaches in both the bedrooms and bathrooms. It made the stay completely unacceptable. Security is also a major issue. The...


RESORT: Swahili Beach Resort
ASSIGNED THEME: Service
--------------------------------------------------
SAMPLE REVIEW TEXT:

## **SECTION K: FETCHING REAL PRICES (APIFY)**

**Objective:** To fulfill Dimension 4 (Pricing & Value), we need real pricing data. The standard Google Places API does not provide base prices, so we had to integrate a third-party web scraper.

**The Solution:** We used the Apify Google Hotels Scraper to fetch the base price (in USD), hotel class (e.g., "4-star hotel"), and property category for our resorts. 

**Methodology & Challenges:** Apify's free tier limits usage to $5.00 per account. To scrape the full dataset of 168 resorts, we divided the scraping process into multiple batches and used separate free accounts to bypass the limit. The scraped data was then merged into our master dataset. A `BasePrice_KES` column was also created using a standard exchange rate (1 USD ≈ 150 KES) for local context.

**Limitations:** While we scraped all 168 resorts, not every resort returned a price. Some smaller guesthouses are not listed on Google Hotels for the selected dates, and some searches returned 0 results. This is a common real-world data gap that is documented in our final dataset.

In [3]:
import os
import pandas as pd
from apify_client import ApifyClient
from dotenv import load_dotenv
import time

# 1. Load the Apify token from .env
load_dotenv()
APIFY_TOKEN = os.getenv("APIFY_API_TOKEN")

if not APIFY_TOKEN:
    raise ValueError("❌ APIFY_API_TOKEN not found in .env file. Check Step 2!")

client = ApifyClient(APIFY_TOKEN)

# 2. Load the dataset
df = pd.read_csv("Data/kenya_resorts_clean.csv")

# 3. Set the limit (Change to None to run all 168 later)
TEST_LIMIT = None  # Set to None to scrape all resorts, or a number for testing 
df_to_scrape = df if TEST_LIMIT is None else df.head(TEST_LIMIT)

print(f"⏳ Scraping prices and amenities for {len(df_to_scrape)} resorts. This will take a few minutes...")

results = []

for index, row in df_to_scrape.iterrows():
    resort_name = row["ResortName"]
    print(f"Scraping: {resort_name}...")
    
    # We use a dummy future date to get a standard base price
    run_input = {
        "destination": f"{resort_name}, Kenya",
        "checkIn": "2026-12-01",  
        "checkOut": "2026-12-02",
        "adults": 2,
        "currency": "USD",
        "maxResultsPerSearch": 1 
    }
    
    try:
        # Run the Actor
        run = client.actor("inexhaustible_glass/google-hotels-scraper").call(run_input=run_input)
        
        # Fetch the results
        item_found = False
        
        # THE FIX IS HERE: changed run["defaultDatasetId"] to run.default_dataset_id
        for item in client.dataset(run.default_dataset_id).iterate_items():
            results.append({
                "ResortName": resort_name,
                "ScrapedPrice": item.get("price"),
                "ScrapedAmenities": ", ".join(item.get("amenities", []))
            })
            item_found = True
            break # Just take the first result
            
        if not item_found:
            results.append({
                "ResortName": resort_name,
                "ScrapedPrice": None,
                "ScrapedAmenities": None
            })
            
        # Pause for a second to respect Apify's rate limits
        time.sleep(2)
        
    except Exception as e:
        print(f"⚠️ Error scraping {resort_name}: {e}")
        results.append({
            "ResortName": resort_name,
            "ScrapedPrice": None,
            "ScrapedAmenities": None
        })

# 4. Show the results
apify_df = pd.DataFrame(results)
print("\n✅ Scraping complete. Here is what we got:")
print(apify_df.to_string())

⏳ Scraping prices and amenities for 168 resorts. This will take a few minutes...
Scraping: Diani Reef Beach Resort & Spa...
⚠️ Error scraping Diani Reef Beach Resort & Spa: POST /v2/actors/inexhaustible_glass~google-hotels-scraper/runs: You've used the $5.00 of monthly usage included in the Free plan, so Actor runs and other Apify services are blocked until your monthly usage cycle resets. GET https://api.apify.com/v2/users/me/limits shows when (monthlyUsageCycle.endAt). The Free plan limit can't be raised. To continue now, upgrade at https://console.apify.com/billing/subscription.
Scraping: Swahili Beach Resort...
⚠️ Error scraping Swahili Beach Resort: POST /v2/actors/inexhaustible_glass~google-hotels-scraper/runs: You've used the $5.00 of monthly usage included in the Free plan, so Actor runs and other Apify services are blocked until your monthly usage cycle resets. GET https://api.apify.com/v2/users/me/limits shows when (monthlyUsageCycle.endAt). The Free plan limit can't be raise

In [1]:
import pandas as pd
import shutil

# 1. Create a backup just in case
shutil.copy("Data/kenya_resorts_clean.csv", "Data/kenya_resorts_clean_BACKUP.csv")
print("✅ Backup created.")

# 2. Load the dataset
df = pd.read_csv("Data/kenya_resorts_clean.csv")

# 3. Identify fake prices (if 3 or more resorts in the same county have the exact same price, it's a median)
same = df.groupby(["County", "BasePrice"])["ResortName"].transform("count") >= 3
print("Likely filled-in prices to remove:", same.sum())

# 4. Blank them out
df.loc[same, ["BasePrice", "BasePrice_KES"]] = None

# 5. Save the corrected file
df.to_csv("Data/kenya_resorts_clean.csv", index=False)

print(f"After cleaning: {df['BasePrice'].notna().sum()} real prices out of 168.")

✅ Backup created.
Likely filled-in prices to remove: 69
After cleaning: 99 real prices out of 168.


**Note on Data Cleaning:** An earlier step filled missing prices with county medians. This cell removes the filled-in values, identified as the same price repeated by 3 or more resorts in a county.

In [2]:
import pandas as pd

# 1. Load the dataset
df = pd.read_csv("Data/kenya_resorts_clean.csv")

# 2. Isiolo and Kajiado had no real prices in the scrape, so these values were filled-in medians
df.loc[df["County"].isin(["Isiolo", "Kajiado"]), ["BasePrice", "BasePrice_KES"]] = None

# 3. Save the corrected file
df.to_csv("Data/kenya_resorts_clean.csv", index=False)

print("Total real prices now:", df["BasePrice"].notna().sum()) # Should print 96

Total real prices now: 96
